In [1]:
import os
import re
import pandas as pd
from pathlib import Path

In [2]:
def clean_step_value(raw_val):
    """
    STEP 파일의 원시 문자열에서 불필요한 기호와 값을 제거하여 깔끔하게 다듬는 함수
    """
    # 1. 따옴표(') 및 NONE 제거
    val = raw_val.replace("'", "")
    val = val.replace("NONE", "")
    
    # 2. 엔티티 참조 ID (예: #123, #45) 제거
    val = re.sub(r'#\d+', '', val)
    
    # 3. 값 제거 후 남은 빈 괄호 ( ) 제거
    val = re.sub(r'\(\s*\)', '', val)
    
    # 4. 쉼표가 여러 개 겹치거나(,,) 시작/끝에 남은 쉼표 및 공백 정리
    val = re.sub(r',\s*,+', ',', val)
    val = val.strip(', ')
    
    # 5. 다중 공백을 하나의 공백으로 압축
    val = re.sub(r'\s+', ' ', val)
    
    return val

def extract_product_metadata_from_step(file_path):
    """STEP 파일에서 지정된 3개의 핵심 PRODUCT 키만 추출 및 정제"""
    info = {'Filename': file_path.name}
    
    # 추출하고자 하는 타겟 키 목록
    target_keys = ['PRODUCT', 'PRODUCT_CONTEXT', 'PRODUCT_DEFINITION_CONTEXT']
    
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            
        content = content.replace('\n', '').replace('\r', '')
        statements = content.split(';')
        
        for stmt in statements:
            stmt = stmt.strip()
            if not stmt.startswith('#'):
                continue
            if 'PRODUCT' not in stmt:
                continue
                
            match = re.match(r'^#\d+\s*=\s*([A-Z0-9_]+)\s*\((.*)\)$', stmt)
            
            if match:
                keyword = match.group(1)
                
                # 타겟 키에 해당하는 경우에만 처리
                if keyword in target_keys:
                    raw_value = match.group(2).strip()
                    cleaned_val = clean_step_value(raw_value)
                    
                    # 정제된 값이 비어있지 않은 경우에만 저장
                    if cleaned_val:
                        # 동일한 키가 여러 개면 ' | ' 기호로 구분하여 연결
                        if keyword in info:
                            info[keyword] += f" | {cleaned_val}"
                        else:
                            info[keyword] = cleaned_val
                            
    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")
        
    return info

def create_filtered_product_dataframe(chunk_dir):
    chunk_path = Path(chunk_dir)
    step_files = list(chunk_path.glob('**/*.step'))
    
    print(f"총 {len(step_files)}개의 STEP 파일을 검색합니다...")
    
    all_data = []
    for i, step_file in enumerate(step_files):
        file_metadata = extract_product_metadata_from_step(step_file)
        all_data.append(file_metadata)
        
        if (i + 1) % 1000 == 0:
            print(f"{i + 1} / {len(step_files)} 개 처리 완료...")
            
    df = pd.DataFrame(all_data)
    
    # 최종적으로 남길 컬럼 리스트
    final_cols = [
        'Filename', 
        'PRODUCT', 
        'PRODUCT_CONTEXT', 
        'PRODUCT_DEFINITION_CONTEXT'
    ]
    
    # 혹시 해당 청크에 특정 키가 아예 없어서 컬럼이 생성되지 않았을 경우를 대비해 빈 컬럼 추가
    for col in final_cols:
        if col not in df.columns:
            df[col] = None
            
    # 지정한 순서대로 컬럼 필터링 및 재배치
    df = df[final_cols]
    
    return df

In [3]:
# 타겟 청크 디렉토리 경로 지정
chunk_num = 3
chunk_num_str = str(chunk_num).zfill(4)
target_chunk_directory = f"./abc_dataset_filtered-1/{chunk_num_str}"

# 데이터프레임 생성 실행
df_products = create_filtered_product_dataframe(target_chunk_directory)

# 데이터프레임 구조 확인
display(df_products.head(10))

# (선택 사항) 결과를 엑셀/CSV로 저장 (한글 및 특수문자 깨짐 방지용 utf-8-sig)
csv_filename = f"./abc_dataset_filtered-1-meta/step_product_metadata_{chunk_num_str}.csv"
df_products.to_csv(csv_filename, index=False, encoding='utf-8-sig')
print(f"추출 완료! '{csv_filename}' 파일로 성공적으로 저장되었습니다.")

총 7319개의 STEP 파일을 검색합니다...
1000 / 7319 개 처리 완료...
2000 / 7319 개 처리 완료...
3000 / 7319 개 처리 완료...
4000 / 7319 개 처리 완료...
5000 / 7319 개 처리 완료...
6000 / 7319 개 처리 완료...
7000 / 7319 개 처리 완료...


,Filename,PRODUCT,PRODUCT_CONTEXT,PRODUCT_DEFINITION_CONTEXT
0,00030000_02de85bae48146e7ae9c883e_step_001.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
1,00030001_02de85bae48146e7ae9c883e_step_002.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
2,00030002_02de85bae48146e7ae9c883e_step_003.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
3,00030003_bb2e6300fc4e4431a3174190_step_000.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
4,00030006_6ae03b5b62304dbea43be8aa_step_000.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
5,00030007_6ae03b5b62304dbea43be8aa_step_001.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
6,00030008_bcedc98b32a44bc39861443a_step_000.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
7,00030009_4f1060c1c16146b79606c36b_step_000.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
8,00030010_0ef34aa1b15748a5b4ad7c0e_step_000.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design
9,00030011_0ef34aa1b15748a5b4ad7c0e_step_001.step,"Part 1, Part 1, PART-Part 1-DESC",mechanical,design


추출 완료! './abc_dataset_filtered-1-meta/step_product_metadata_0003.csv' 파일로 성공적으로 저장되었습니다.
